<a href="https://colab.research.google.com/github/sdbileg/alpha/blob/main/reconcile_sandbox.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import io

# 1. Internal Feed: Single continuous text block separated by inconsistent multi-spaces/tabs
internal_raw_text = """
ACCOUNT_ID      SYMBOL    CUSIP        PRE_SPLIT_QTY
ACC-101         XYZ       987654101    105
ACC-102         XYZ       987654101    200
ACC-103         XYZ       087654101    33
"""

# 2. Custodian/DTC Feed: Raw space-delimited extract with leading zeroes stripped on CUSIPs
external_raw_text = """
ACCOUNT_NUM CUSIP DTC_SHARE_QTY DTC_CASH_CREDIT
ACC-101 987654101 157 75.00
ACC-102 987654101 300 0.00
ACC-103 87654101 49 70.00
"""

print("--- UNPARSED INTERNAL RAW TEXT ---")
print(internal_raw_text)

print("--- UNPARSED CUSTODIAN RAW TEXT ---")
print(external_raw_text)

--- UNPARSED INTERNAL RAW TEXT ---

ACCOUNT_ID      SYMBOL    CUSIP        PRE_SPLIT_QTY
ACC-101         XYZ       987654101    105
ACC-102         XYZ       987654101    200
ACC-103         XYZ       087654101    33

--- UNPARSED CUSTODIAN RAW TEXT ---

ACCOUNT_NUM CUSIP DTC_SHARE_QTY DTC_CASH_CREDIT
ACC-101 987654101 157 75.00
ACC-102 987654101 300 0.00
ACC-103 87654101 49 70.00



In [ ]:
def parse_messy_text_feed(raw_text_data):
    """
    Parses continuous multi-space/whitespace delimited raw text into a clean DataFrame.
    """
    # read_csv with regex engine handling variable whitespace (sep=r'\s+')
    df = pd.read_csv(
        io.StringIO(raw_text_data.strip()),
        sep=r'\s+',              # Delimiter: 1 or more spaces/tabs
        engine='python',         # Python engine required for regex separators
        dtype=str                # Read all as strings initially to preserve leading zeroes
    )
    return df

# Ingest and align datasets
df_internal_raw = parse_messy_text_feed(internal_raw_text)
df_external_raw = parse_messy_text_feed(external_raw_text)

print("Aligned Internal DataFrame:")
display(df_internal_raw)

print("\nAligned External DataFrame:")
display(df_external_raw)

Aligned Internal DataFrame:


,ACCOUNT_ID,SYMBOL,CUSIP,PRE_SPLIT_QTY
0,ACC-101,XYZ,987654101,105
1,ACC-102,XYZ,987654101,200
2,ACC-103,XYZ,087654101,33



Aligned External DataFrame:


,ACCOUNT_NUM,CUSIP,DTC_SHARE_QTY,DTC_CASH_CREDIT
0,ACC-101,987654101,157,75.00
1,ACC-102,987654101,300,0.00
2,ACC-103,87654101,49,70.00


In [ ]:
def sanitize_parsed_data(df, account_col, cusip_col, numeric_cols):
    """Sanitizes aligned columns, fixes CUSIP padding, and casts data types."""
    df = df.copy()

    # Strip whitespace from string keys
    df[account_col] = df[account_col].str.strip()

    # Enforce standard 9-digit CUSIP length (restores dropped leading 0s)
    df[cusip_col] = df[cusip_col].str.strip().str.zfill(9)

    # Convert numerical strings to float/int
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].str.replace('$', '').str.replace(',', ''), errors='coerce').fillna(0)

    return df

# Sanitize Data
df_internal = sanitize_parsed_data(
    df_internal_raw,
    account_col='ACCOUNT_ID',
    cusip_col='CUSIP',
    numeric_cols=['PRE_SPLIT_QTY']
)

df_external = sanitize_parsed_data(
    df_external_raw,
    account_col='ACCOUNT_NUM',
    cusip_col='CUSIP',
    numeric_cols=['DTC_SHARE_QTY', 'DTC_CASH_CREDIT']
)

print("Final Sanitized Internal Data (Notice CUSIP 087654101 leading zero restored):")
display(df_internal)

Final Sanitized Internal Data (Notice CUSIP 087654101 leading zero restored):


,ACCOUNT_ID,SYMBOL,CUSIP,PRE_SPLIT_QTY
0,ACC-101,XYZ,987654101,105
1,ACC-102,XYZ,987654101,200
2,ACC-103,XYZ,087654101,33


In [ ]:
# Corporate Action Parameters (3:2 Stock Split @ $150 CIL Price)
SPLIT_RATIO = 1.5
CIL_PRICE = 150.00

# Merge and reconcile
recon = pd.merge(
    df_internal,
    df_external,
    left_on=['ACCOUNT_ID', 'CUSIP'],
    right_on=['ACCOUNT_NUM', 'CUSIP'],
    how='outer'
)

# Recon Calculations
recon['raw_entitled_qty'] = recon['PRE_SPLIT_QTY'] * SPLIT_RATIO
recon['expected_whole_shares'] = np.floor(recon['raw_entitled_qty']).astype(int)
recon['fractional_remainder'] = np.round(recon['raw_entitled_qty'] - recon['expected_whole_shares'], 4)
recon['expected_cil_cash'] = recon['fractional_remainder'] * CIL_PRICE

recon['share_delta'] = recon['expected_whole_shares'] - recon['DTC_SHARE_QTY']
recon['cash_delta'] = recon['expected_cil_cash'] - recon['DTC_CASH_CREDIT']

# Classify Status
conditions = [
    (recon['share_delta'] == 0) & (recon['cash_delta'] == 0),
    (recon['share_delta'] == 0) & (recon['cash_delta'].abs() > 0.01)
]
recon['recon_status'] = np.select(conditions, ['Matched', 'CIL Cash Variance'], default='Quantity Break')

display(recon[['ACCOUNT_ID', 'CUSIP', 'recon_status', 'expected_whole_shares', 'DTC_SHARE_QTY', 'expected_cil_cash', 'DTC_CASH_CREDIT', 'cash_delta']])

,ACCOUNT_ID,CUSIP,recon_status,expected_whole_shares,DTC_SHARE_QTY,expected_cil_cash,DTC_CASH_CREDIT,cash_delta
0,ACC-101,987654101,Matched,157,157,75.0,75.0,0.0
1,ACC-102,987654101,Matched,300,300,0.0,0.0,0.0
2,ACC-103,087654101,CIL Cash Variance,49,49,75.0,70.0,5.0


In [ ]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

def export_recon_to_excel(recon_df, output_filename="Brokerage_Ops_Reconciliation_Report.xlsx"):
    """
    Exports reconciliation DataFrame into a polished multi-tab Excel report
    with conditional formatting for operational breaks and metric dashboards.
    """
    wb = openpyxl.Workbook()

    # -------------------------------------------------------------
    # TAB 1: EXECUTIVE DASHBOARD
    # -------------------------------------------------------------
    ws_summary = wb.active
    ws_summary.title = "Executive Summary"
    ws_summary.views.sheetView[0].showGridLines = True

    # Header Banner
    ws_summary.merge_cells("A1:E1")
    title_cell = ws_summary["A1"]
    title_cell.value = "BROKERAGE OPERATIONS RECONCILIATION DASHBOARD"
    title_cell.font = Font(name="Segoe UI", size=13, bold=True, color="FFFFFF")
    title_cell.fill = PatternFill(start_color="1F497D", end_color="1F497D", fill_type="solid")
    title_cell.alignment = Alignment(horizontal="center", vertical="center")
    ws_summary.row_dimensions[1].height = 35

    # Metrics Table Setup
    ws_summary.cell(row=3, column=1, value="Key Performance Indicators").font = Font(name="Segoe UI", size=11, bold=True, color="1F497D")

    total_records = len(recon_df)
    matched_records = len(recon_df[recon_df['recon_status'] == 'Matched'])

    kpi_data = [
        ("Corporate Action Event", "3:2 Stock Split (XYZ Corp)"),
        ("Total Accounts Evaluated", total_records),
        ("Matched Positions", matched_records),
        ("Total Exceptions", total_records - matched_records),
        ("Match Rate", f"{(matched_records / total_records) * 100:.1f}%"),
        ("Total Cash Exposure ($", recon_df['cash_delta'].abs().sum())
    ]

    for row_idx, (metric, val) in enumerate(kpi_data, start=4):
        ws_summary.cell(row=row_idx, column=1, value=metric).font = Font(name="Segoe UI", bold=True)
        val_cell = ws_summary.cell(row=row_idx, column=2, value=val)
        val_cell.font = Font(name="Segoe UI")
        if isinstance(val, (int, float)):
            val_cell.number_format = '$#,##0.00' if 'Exposure' in metric else '#,##0'

    # -------------------------------------------------------------
    # TAB 2: RECONCILIATION DETAIL TABLE
    # -------------------------------------------------------------
    ws_detail = wb.create_sheet(title="Reconciliation Detail")
    ws_detail.views.sheetView[0].showGridLines = True

    # Append Header Row
    ws_detail.append(list(recon_df.columns))
    header_fill = PatternFill(start_color="1F497D", end_color="1F497D", fill_type="solid")
    header_font = Font(name="Segoe UI", size=10, bold=True, color="FFFFFF")

    for cell in ws_detail[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws_detail.row_dimensions[1].height = 28

    # Table Grid Styles
    thin_border = Border(
        left=Side(style='thin', color='E0E0E0'),
        right=Side(style='thin', color='E0E0E0'),
        top=Side(style='thin', color='E0E0E0'),
        bottom=Side(style='thin', color='E0E0E0')
    )
    zebra_fill = PatternFill(start_color="F9FAFB", end_color="F9FAFB", fill_type="solid")

    # Populate Data Rows
    for r_idx, row in recon_df.iterrows():
        row_num = r_idx + 2
        ws_detail.append(list(row))
        for c_idx, val in enumerate(row, start=1):
            cell = ws_detail.cell(row=row_num, column=c_idx)
            cell.font = Font(name="Segoe UI", size=10)
            cell.border = thin_border

            # Subtle zebra striping on alternating rows
            if row_num % 2 == 0:
                cell.fill = zebra_fill

            # Column Formatting & Alignment
            col_name = recon_df.columns[c_idx-1]
            if "qty" in col_name.lower() or "shares" in col_name.lower() or "delta" in col_name.lower():
                cell.alignment = Alignment(horizontal="right")
                cell.number_format = '$#,##0.00' if "cash_delta" in col_name.lower() else '#,##0'
            elif col_name in ['ACCOUNT_ID', 'CUSIP', 'SYMBOL', 'ACCOUNT_NUM']:
                cell.alignment = Alignment(horizontal="center")

    # -------------------------------------------------------------
    # CONDITIONAL FORMATTING HIGHLIGHTS
    # -------------------------------------------------------------
    # Color Fills
    green_fill = PatternFill(start_color="D4EDDA", end_color="D4EDDA", fill_type="solid")
    green_font = Font(name="Segoe UI", color="155724", bold=True)

    red_fill = PatternFill(start_color="F8D7DA", end_color="F8D7DA", fill_type="solid")
    red_font = Font(name="Segoe UI", color="721C24", bold=True)

    yellow_fill = PatternFill(start_color="FFF3CD", end_color="FFF3CD", fill_type="solid")
    yellow_font = Font(name="Segoe UI", color="856404", bold=True)

    status_col_idx = list(recon_df.columns).index('recon_status') + 1
    share_delta_col_idx = list(recon_df.columns).index('share_delta') + 1
    cash_delta_col_idx = list(recon_df.columns).index('cash_delta') + 1

    for row in range(2, len(recon_df) + 2):
        status_cell = ws_detail.cell(row=row, column=status_col_idx)
        share_delta_cell = ws_detail.cell(row=row, column=share_delta_col_idx)
        cash_delta_cell = ws_detail.cell(row=row, column=cash_delta_col_idx)

        # Highlight Status Column
        if status_cell.value == 'Matched':
            status_cell.fill = green_fill
            status_cell.font = green_font
        elif 'Variance' in str(status_cell.value):
            status_cell.fill = yellow_fill
            status_cell.font = yellow_font
        else:
            status_cell.fill = red_fill
            status_cell.font = red_font

        # Highlight Delta Columns on Exception
        if share_delta_cell.value != 0:
            share_delta_cell.fill = red_fill
            share_delta_cell.font = red_font
        if abs(cash_delta_cell.value) > 0.01:
            cash_delta_cell.fill = yellow_fill
            cash_delta_cell.font = yellow_font

    # Auto-fit Column Widths
    for ws in [ws_summary, ws_detail]:
        for col in ws.columns:
            max_len = max(len(str(cell.value or '')) for cell in col)
            col_letter = get_column_letter(col[0].column)
            ws.column_dimensions[col_letter].width = max(max_len + 4, 14)

    wb.save(output_filename)
    print(f"Report successfully saved to {output_filename}")

# Run export function
export_recon_to_excel(recon)

Report successfully saved to Brokerage_Ops_Reconciliation_Report.xlsx
